# Qubitized Hubbard

Simulating the Hubbard model Hamiltonian using qubitization.

This module follows section V. of Encoding Electronic Spectra in Quantum Circuits with Linear T
Complexity. Babbush et al. 2018. [arxiv:1805.03662](https://arxiv.org/abs/1805.03662).

The 2D Hubbard model is a special case of the electronic structure Hamiltonian
restricted to spins on a planar grid.

$$
H = -t \sum_{\langle p,q \rangle, \sigma} a_{p,\sigma}^\dagger a_{q,\sigma}
    + \frac{u}{2} \sum_{p,\alpha\ne\beta} n_{p, \alpha} n_{p, \beta}
$$

Under the Jordan-Wigner transformation to Pauli operators, this is

$$
\def\Zvec{\overrightarrow{Z}}
\def\hop#1{#1_{p,\sigma} \Zvec #1_{q,\sigma}}
H = -\frac{t}{2} \sum_{\langle p,q \rangle, \sigma} (\hop{X} + \hop{Y})
  + \frac{u}{8} \sum_{p,\alpha\ne\beta} Z_{p,\alpha}Z_{p,\beta}
  - \frac{u}{4} \sum_{p,\sigma} Z_{p,\sigma} + \frac{uN}{4}\mathbb{1}
$$

This can be simulated using a qubitization circuit, which consists of PREPARE and SELECT
operations. This module contains `SelectHubbard` and `PrepareHubbard`, with particular
compilation optimizations for the Hubbard model. For more insight into how Select and Prepare
operations can be combined into a quantum walk, please see
[Qubitization Walk Operator](./qubitization_walk_operator.ipynb).

With these operators, our selection register has indices
for $p$, $\alpha$, $q$, and $\beta$ as well as two indicator bits $U$ and $V$. There are four cases
considered in both the PREPARE and SELECT operations corresponding to the terms in the Hamiltonian:

 - $U=1$, single-body Z
 - $V=1$, spin-spin ZZ term
 - $p<q$, XZX term
 - $p>q$, YZY term.
Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.


In [ ]:
from dataclasses import asdict

from IPython.display import SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.hubbard_qubitization_resources import (
    notebook_models, notebook_signature,
)
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg
from flagquantum.qec.resources.estimation import estimate_ftqc_resources


def show_resource_graph(model):
    dot = export_modeled_call_graph_dot(model, max_depth=1)
    display(SVG(render_call_graph_svg(dot)))
    report = estimate_ftqc_resources(model)
    assert report.emitted is None
    print("Logical gate estimate", asdict(report.logical.gates))
    print("Peak logical qubits", report.logical.peak_logical_qubits)


## `SelectHubbard`

SELECT acts on the system indexed by the two coordinate and spin addresses.
It composes two Majorana operations, swaps the selection coordinates where
required, corrects the Pauli product phase, and applies the spin-up Z term.
The target order is spin, y, then x with x varying fastest.

In [ ]:
from flagquantum.algorithms.primitives.hubbard_select import SelectHubbard

### Example Instances

In [ ]:
select = SelectHubbard(4)
select_signature = notebook_signature(4)
_, select_model = notebook_models()[0]
assert select.source_resource_model() == select_model

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("4×4 SELECT", select_signature, {})])))

### Call Graph

In [ ]:
show_resource_graph(select_model)

## `PrepareHubbard`

PREPARE places the selection words in a superposition with amplitudes
determined by the hopping and interaction coefficients. Its two direction
bits in `temp` are part of the logical output. The work qubits used by
uniform preparation, modular shifts, and measurement feedback are separate.

In [ ]:
from flagquantum.algorithms.primitives.hubbard_prepare import PrepareHubbard

### Example Instances

In [ ]:
prepare = PrepareHubbard(4, t=1.0, u=4.0)
prepare_signature = notebook_signature(4, prepare=True)
_, prepare_model = notebook_models()[1]
assert prepare.source_resource_model() == prepare_model

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("4×4 PREPARE", prepare_signature, {})])))

### Call Graph

In [ ]:
show_resource_graph(prepare_model)